In [2]:
%env KAGGLE_KEY=KGAT_345ff9f1af4cf2fa28ea0a3cab3fd477
%env KAGGLE_USERNAME=luizmonteiro
import kagglehub
import pandas as pd

env: KAGGLE_KEY=KGAT_345ff9f1af4cf2fa28ea0a3cab3fd477
env: KAGGLE_USERNAME=luizmonteiro


d:\Anaconda\envs\tcc_faturas\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
path = kagglehub.dataset_download('nalisha/netflix-movies-and-tv-shows-data-analysis', output_dir='../data/kaggle')
print("Path to competition files:", path)

In [2]:
# Download latest version
patholist = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", patholist)

Path to dataset files: C:\Users\DEVELOPER\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2


In [ ]:
data = pd.read_csv(f'{patholist}/olist_order_items_dataset.csv')
data.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


: 

In [3]:
transactions = kagglehub.dataset_download("ealtman2019/credit-card-transactions")

print("Path to dataset files:", transactions)

100%|██████████| 263M/263M [00:10<00:00, 27.1MB/s] 


Extracting files...
Path to dataset files: C:\Users\DEVELOPER\.cache\kagglehub\datasets\ealtman2019\credit-card-transactions\versions\8


## Combinador de CSVs em ZIPs

Esta função extrai todos os arquivos CSV de múltiplos ZIPs e os combina em um único CSV.

### Uso:
```python
df = combinar_csvs_de_zips(
    pasta_zips='./dados_zips',      # Pasta contendo os ZIPs
    pasta_saida='./dados_saida',    # Pasta onde salvar o resultado
    arquivo_saida='resultado.csv'   # Nome do arquivo final
)
```

### Recursos:
- ✓ Extrai CSVs automaticamente de todos os ZIPs
- ✓ Combina todos em um único DataFrame
- ✓ Salva em CSV com encoding UTF-8
- ✓ Mostra progresso e estatísticas
- ✓ Trata erros individualmente

In [22]:
import zipfile
import os
from pathlib import Path
import pandas as pd
def combinar_csvs_de_zips(pasta_zips, pasta_saida=None, arquivo_saida='dados_combinados.csv'):
    """
    Extrai todos os CSVs de arquivos ZIP em uma pasta e os combina em um único CSV.
    
    Parameters:
    -----------
    pasta_zips : str
        Caminho da pasta contendo os arquivos ZIP
    pasta_saida : str, optional
        Caminho da pasta de saída. Se None, cria em 'dados_combinados'
    arquivo_saida : str, default='dados_combinados.csv'
        Nome do arquivo CSV de saída
    
    Returns:
    --------
    pd.DataFrame
        DataFrame com todos os dados combinados
    """
    
    if pasta_saida is None:
        pasta_saida = 'dados_combinados'
    
    Path(pasta_saida).mkdir(exist_ok=True)
    
    dataframes = []
    contador = 0
    
    arquivos_zip = list(Path(pasta_zips).glob('*.zip'))
    
    if not arquivos_zip:
        print(f"Nenhum arquivo ZIP encontrado em {pasta_zips}")
        return None
    
    print(f"Encontrados {len(arquivos_zip)} arquivo(s) ZIP")
    
    for zip_file in arquivos_zip:
        print(f"\nProcessando: {zip_file.name}")
        
        try:
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                arquivos_csv = [f for f in zip_ref.namelist() if f.endswith('.csv')]
                
                if not arquivos_csv:
                    print(f"  ⚠️  Nenhum CSV encontrado neste ZIP")
                    continue
                
                print(f"  Encontrados {len(arquivos_csv)} arquivo(s) CSV")
                
                for csv_file in arquivos_csv:
                    try:
                        with zip_ref.open(csv_file) as f:
                            df = pd.read_csv(f, sep=';', encoding='latin-1')
                            dataframes.append(df)
                            contador += 1
                            print(f"    ✓ {csv_file} ({len(df)} linhas)")
                    except Exception as e:
                        print(f"    ✗ Erro ao ler {csv_file}: {e}")
        
        except Exception as e:
            print(f"  Erro ao processar ZIP: {e}")
    
    if not dataframes:
        print("\n❌ Nenhum CSV foi extraído!")
        return None
    
    print(f"\n📊 Combinando {contador} arquivo(s) CSV...")
    df_final = pd.concat(dataframes, ignore_index=True)
    
    caminho_saida = os.path.join(pasta_saida, arquivo_saida)
    df_final.to_csv(caminho_saida, index=False, encoding='utf-8')
    
    print(f"✅ Arquivo salvo em: {caminho_saida}")
    print(f"   Linhas totais: {len(df_final)}")
    print(f"   Colunas: {list(df_final.columns)}")
    
    return df_final

In [23]:
import random
import string

def simular_nome_maquininha_cartao(nome, comprimento_max=20):
    """
    Simula como o nome de um fornecedor aparece em uma descrição de máquina de cartão.
    Adiciona ruído realista como truncamento, códigos e caracteres estranhos.
    
    Parameters:
    -----------
    nome : str
        Nome original do fornecedor
    comprimento_max : int
        Comprimento máximo permitido pela máquina
    
    Returns:
    --------
    str
        Nome com ruído simulado
    """
    
    if pd.isna(nome):
        return nome
    
    nome_str = str(nome).upper().strip()
    
    truncado = nome_str[:comprimento_max]
    
    tipos_ruido = random.choices(
        [0, 1, 2, 3, 4],
        weights=[30, 25, 20, 15, 10],
        k=1
    )[0]
    
    if tipos_ruido == 0:
        sufixo = f" {random.randint(100, 9999)}"
        resultado = (truncado[:comprimento_max-len(sufixo)] + sufixo).rstrip()
    
    elif tipos_ruido == 1:
        prefixo_loja = f"*{random.randint(10, 9999)} "
        resultado = (prefixo_loja + truncado[:comprimento_max-len(prefixo_loja)]).rstrip()
    
    elif tipos_ruido == 2:
        partes = truncado.split()
        if len(partes) > 1:
            resultado = partes[0] + " " + partes[1][:3].upper()
        else:
            resultado = truncado
    
    elif tipos_ruido == 3:
        caracteres_aleatorios = ''.join(random.choices(string.ascii_uppercase + string.digits, k=3))
        resultado = truncado[:comprimento_max-4] + caracteres_aleatorios
    
    else:
        resultado = truncado
    
    return resultado.strip()

def gerar_dataset_com_ruido(df, coluna_nome='NOME FAVORECIDO', probabilidade=1.0):
    """
    Adiciona coluna com nomes simulados de máquina de cartão.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame com os dados
    coluna_nome : str
        Nome da coluna com os fornecedores
    probabilidade : float
        Probabilidade de adicionar ruído (0-1)
    
    Returns:
    --------
    pd.DataFrame
        DataFrame com nova coluna 'DESCRICAO_MAQUININHA'
    """
    
    df_novo = df.copy()
    
    df_novo['DESCRICAO_MAQUININHA'] = df_novo[coluna_nome].apply(
        lambda x: simular_nome_maquininha_cartao(x) if random.random() < probabilidade else x
    )
    
    print(f"✅ Coluna 'DESCRICAO_MAQUININHA' criada com ruído de máquina de cartão!")
    print(f"\nExemplos de transformação:")
    
    comparacao = df_novo[[coluna_nome, 'DESCRICAO_MAQUININHA']].drop_duplicates().head(10)
    for idx, row in comparacao.iterrows():
        print(f"  {row[coluna_nome]:30s} → {row['DESCRICAO_MAQUININHA']:25s}")
    
    return df_novo

print("Função de simulação de máquina de cartão criada!")
print("\nUso:")
print("df_com_ruido = gerar_dataset_com_ruido(df_combinado)")
print("\nOu com probabilidade customizada:")
print("df_com_ruido = gerar_dataset_com_ruido(df_combinado, probabilidade=0.8)")

Função de simulação de máquina de cartão criada!

Uso:
df_com_ruido = gerar_dataset_com_ruido(df_combinado)

Ou com probabilidade customizada:
df_com_ruido = gerar_dataset_com_ruido(df_combinado, probabilidade=0.8)


In [16]:
def filtrar_sem_nan(df, colunas=None, como='dropna'):
    """
    Filtra DataFrame removendo linhas com NaN.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame a filtrar
    colunas : list, optional
        Lista de colunas específicas a verificar. Se None, verifica todas
    como : str, default='dropna'
        'dropna' = remove linhas com NaN
        'fill_zero' = substitui NaN por 0
        'fill_mean' = substitui NaN pela média da coluna
        'fill_forward' = propaga valor anterior
    
    Returns:
    --------
    pd.DataFrame
        DataFrame filtrado
    """
    
    df_filtrado = df.copy()
    
    if como == 'dropna':
        if colunas:
            df_filtrado = df_filtrado.dropna(subset=colunas)
        else:
            df_filtrado = df_filtrado.dropna()
        
        print(f"Linhas removidas: {len(df) - len(df_filtrado)}")
        print(f"Linhas restantes: {len(df_filtrado)}")
    
    elif como == 'fill_zero':
        df_filtrado = df_filtrado.fillna(0)
        print(f"NaNs substituídos por 0")
    
    elif como == 'fill_mean':
        if colunas:
            for col in colunas:
                df_filtrado[col] = df_filtrado[col].fillna(df_filtrado[col].mean())
        else:
            df_filtrado = df_filtrado.fillna(df_filtrado.mean())
        print(f"NaNs substituídos pela média")
    
    elif como == 'fill_forward':
        df_filtrado = df_filtrado.fillna(method='ffill')
        print(f"NaNs substituídos pela propagação anterior")
    
    return df_filtrado

print("Função de filtro de NaN criada!")
print("\nExemplos de uso:")
print("1. Remover todas as linhas com NaN:")
print("   df_limpo = filtrar_sem_nan(df)")
print("\n2. Remover apenas linhas onde colunas específicas têm NaN:")
print("   df_limpo = filtrar_sem_nan(df, colunas=['coluna1', 'coluna2'])")
print("\n3. Preencher NaN com 0:")
print("   df_preenchido = filtrar_sem_nan(df, como='fill_zero')")
print("\n4. Preencher NaN com a média:")
print("   df_media = filtrar_sem_nan(df, como='fill_mean')")

Função de filtro de NaN criada!

Exemplos de uso:
1. Remover todas as linhas com NaN:
   df_limpo = filtrar_sem_nan(df)

2. Remover apenas linhas onde colunas específicas têm NaN:
   df_limpo = filtrar_sem_nan(df, colunas=['coluna1', 'coluna2'])

3. Preencher NaN com 0:
   df_preenchido = filtrar_sem_nan(df, como='fill_zero')

4. Preencher NaN com a média:
   df_media = filtrar_sem_nan(df, como='fill_mean')


In [25]:
df_combinado = combinar_csvs_de_zips(
    pasta_zips='../data/datasets/',  # Altere para sua pasta com ZIPs
    pasta_saida='../data/datasets/',
    arquivo_saida='faturas_gov.csv'
)


Encontrados 43 arquivo(s) ZIP

Processando: 202301_CPGF.zip
  Encontrados 1 arquivo(s) CSV
    ✓ 202301_CPGF.csv (10439 linhas)

Processando: 202302_CPGF.zip
  Encontrados 1 arquivo(s) CSV
    ✓ 202302_CPGF.csv (1833 linhas)

Processando: 202303_CPGF.zip
  Encontrados 1 arquivo(s) CSV
    ✓ 202303_CPGF.csv (6432 linhas)

Processando: 202304_CPGF.zip
  Encontrados 1 arquivo(s) CSV
    ✓ 202304_CPGF.csv (11184 linhas)

Processando: 202305_CPGF.zip
  Encontrados 1 arquivo(s) CSV
    ✓ 202305_CPGF.csv (9964 linhas)

Processando: 202306_CPGF.zip
  Encontrados 1 arquivo(s) CSV
    ✓ 202306_CPGF.csv (13509 linhas)

Processando: 202307_CPGF.zip
  Encontrados 1 arquivo(s) CSV
    ✓ 202307_CPGF.csv (11669 linhas)

Processando: 202308_CPGF.zip
  Encontrados 1 arquivo(s) CSV
    ✓ 202308_CPGF.csv (12448 linhas)

Processando: 202309_CPGF.zip
  Encontrados 1 arquivo(s) CSV
    ✓ 202309_CPGF.csv (13495 linhas)

Processando: 202310_CPGF.zip
  Encontrados 1 arquivo(s) CSV
    ✓ 202310_CPGF.csv (12027 l

#### Gov Dataset https://portaldatransparencia.gov.br/download-de-dados/cpgf

In [20]:
df_sem_nulos = filtrar_sem_nan(df_combinado, colunas=['CPF'])
df_sem_nulos.head()

Linhas removidas: 477242
Linhas restantes: 498


,CÓDIGO ÓRGÃO SUPERIOR,NOME ÓRGÃO SUPERIOR,CÓDIGO ÓRGÃO,NOME ÓRGÃO,CÓDIGO UNIDADE GESTORA,NOME UNIDADE GESTORA,ANO EXTRATO,MÊS EXTRATO,CPF PORTADOR,NOME PORTADOR,...,VALOR TRANSAÇÃO,TIPO AQUISIÇÃO,ENDEREÇO,ÓRGÃO RESPONSÁVEL,SITUAÇÃO,NOME PERMISSIONÁRIO,CPF,CARGO OU FUNÇÃO DE CONFIANÇA,ÓRGÃO EXERCÍCIO DO PERMISSIONÁRIO,DATA INÍCIO OCUPAÇÃO
442562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,ACIR PIMENTA MADEIRA FILHO,***.123.906-**,MINISTRO DE SEGUNDA CLASSE,MINISTÉRIO DAS RELAÇÕES EXTERIORES / MRE,01/09/2022
442563,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,ADRIANA DE MEDEIROS GABINIO,***.035.174-**,TERCEIRO SECRETÁRIO,MINISTÉRIO DAS RELAÇÕES EXTERIORES / MRE,01/12/2023
442564,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,ADRIANA FERNANDES FARIAS,***.870.928-**,PRIMEIRO SECRETÁRIO,MINISTÉRIO DAS RELAÇÕES EXTERIORES / MRE,01/05/2024
442565,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,ADRIANA NEVES LOVIS,***.546.361-**,ASSISTENTE DE CHANCELARIA,MINISTÉRIO DAS RELAÇÕES EXTERIORES / MRE,01/06/2006
442566,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,ADRIANA PEREIRA DE CASTRO FERREIRA,***.648.318-**,OFICIAL DE CHANCELARIA,MINISTÉRIO DAS RELAÇÕES EXTERIORES / MRE,01/08/2024


In [21]:
df_combinado['NOME FAVORECIDO']

0                                   IMPORTADORA OPLIMA LTDA
1                            JOSE PEREIRA DE SOUZA MOLDURAS
2                                  EXTINTORES ARAGUAIA LTDA
3                                      SACARIA ESTRELA LTDA
4         BIG CHAVES COMERCIO E SERVICOS DE CHAVES, CARI...
                                ...                        
477735                         AUTO POSTO HYGIENOPOLIS LTDA
477736                         AUTO POSTO HYGIENOPOLIS LTDA
477737                         AUTO POSTO HYGIENOPOLIS LTDA
477738                         AUTO POSTO HYGIENOPOLIS LTDA
477739                 POSTO DE SERVICO JARDIM AMERICA LTDA
Name: NOME FAVORECIDO, Length: 477740, dtype: object

In [ ]:
df_com_ruido = gerar_dataset_com_ruido(df_combinado)
    
print("\nAmostra do resultado:")
print(df_com_ruido[['NOME FAVORECIDO', 'DESCRICAO_MAQUININHA']].head(15))

✅ Coluna 'DESCRICAO_MAQUININHA' criada com ruído de máquina de cartão!

Exemplos de transformação:
  IMPORTADORA OPLIMA LTDA        → IMPORTADORA OPL          
  JOSE PEREIRA DE SOUZA MOLDURAS → *7384 JOSE PEREIRA D     
  EXTINTORES ARAGUAIA LTDA       → EXTINTORES ARA           
  SACARIA ESTRELA LTDA           → SACARIA ESTRELA 2744     
  BIG CHAVES COMERCIO E SERVICOS DE CHAVES, CARIMBOS E SISTEMA DE SEGURANCA LTDA → BIG CHAVES COME 8843     
  JANIO C. C. DA SILVA           → JANIO C. C. DA SILVA     
  IG PLOTTER GRAFICA RAPIDA LTDA → IG PLO                   
  FUJIOKA ELETRO IMAGEM S.A      → FUJIOKA ELETRO IMAGE     
  OPCAO FERRAZ FERRAGISTA LTDA   → OPCAO FER                
  OPCAO FERRAZ FERRAGISTA LTDA   → OPCAO FERRAZ FE 6785     

Amostra do resultado:
                                      NOME FAVORECIDO  DESCRICAO_MAQUININHA
0                             IMPORTADORA OPLIMA LTDA       IMPORTADORA OPL
1                      JOSE PEREIRA DE SOUZA MOLDURAS  *7384 JOSE PE

In [1]:
import pandas as pd

In [6]:
df = pd.read_excel('../data/datasets/tabelas_2017_2018_xls/Tabela 1b.xlsx')

In [7]:
df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,Tabela 1b - Proporção de pessoas das famílias ...,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,Condicionantes e subgrupos\nselecionados,Proporção de pessoas das famílias residentes (%),Proporção de pessoas com algum grau de vulnera...,IVM-NM,Contribuição para o IVM-NM do Brasil,Contribuição para o IVM-NM do Brasil (%)
3,Localização geográfica do domicílio,NaN,NaN,NaN,NaN,NaN
4,Brasil,100,63.810533,7.69625,7.69625,100


In [2]:
import glob
import pandas as pd

def combinar_csvs_simples(pasta, padrao='*.csv', arquivo_saida=None):
    """
    Combina múltiplos CSVs de uma pasta em um único DataFrame.
    
    Parameters:
    -----------
    pasta : str
        Caminho da pasta contendo os arquivos CSV
    padrao : str, default='*.csv'
        Padrão de busca dos arquivos (ex: 'dados_*.csv')
    arquivo_saida : str, optional
        Se fornecido, salva o resultado em CSV
    
    Returns:
    --------
    pd.DataFrame
        DataFrame com todos os dados combinados
    """
    
    caminho_busca = f'{pasta}/{padrao}'
    arquivos_csv = glob.glob(caminho_busca)
    
    if not arquivos_csv:
        print(f"❌ Nenhum arquivo encontrado em: {caminho_busca}")
        return None
    
    print(f"✅ Encontrados {len(arquivos_csv)} arquivo(s) CSV\n")
    
    dataframes = []
    for arquivo in sorted(arquivos_csv):
        try:
            df = pd.read_csv(arquivo)
            dataframes.append(df)
            print(f"  ✓ {arquivo.split('/')[-1]:30s} - {len(df):6d} linhas")
        except Exception as e:
            print(f"  ✗ Erro ao ler {arquivo}: {e}")
    
    if not dataframes:
        print("\n❌ Nenhum CSV foi carregado!")
        return None
    
    print(f"\n📊 Combinando dados...")
    df_final = pd.concat(dataframes, ignore_index=True)
    
    print(f"✅ Total: {len(df_final)} linhas × {len(df_final.columns)} colunas")
    
    if arquivo_saida:
        df_final.to_csv(arquivo_saida, index=False, encoding='utf-8')
        print(f"📁 Arquivo salvo em: {arquivo_saida}")
    
    return df_final

print("✅ Função de combinação de CSVs criada!")
print("\nExemplo de uso:")
print("df = combinar_csvs_simples('dados/', arquivo_saida='resultado.csv')")
print("df = combinar_csvs_simples('dados/', padrao='vendas_*.csv')")


✅ Função de combinação de CSVs criada!

Exemplo de uso:
df = combinar_csvs_simples('dados/', arquivo_saida='resultado.csv')
df = combinar_csvs_simples('dados/', padrao='vendas_*.csv')


In [ ]:
df = combinar_csvs_simples('../data/datasets/C6/', arquivo_saida='gastosc6.csv')

✅ Encontrados 18 arquivo(s) CSV

  ✓ C6\Fatura_2024-01-15.csv       -     51 linhas
  ✓ C6\Fatura_2024-02-15.csv       -     41 linhas
  ✓ C6\Fatura_2024-03-15.csv       -     37 linhas
  ✓ C6\Fatura_2024-04-15.csv       -     25 linhas
  ✓ C6\Fatura_2025-01-15.csv       -     91 linhas
  ✓ C6\Fatura_2025-02-15.csv       -     67 linhas
  ✓ C6\Fatura_2025-03-15.csv       -     61 linhas
  ✓ C6\Fatura_2025-04-15.csv       -     74 linhas
  ✗ Erro ao ler ../data/datasets/C6\Fatura_2025-05-15.csv: Error tokenizing data. C error: Expected 1 fields in line 82, saw 2

  ✓ C6\Fatura_2025-09-15.csv       -     91 linhas
  ✓ C6\Fatura_2025-10-15.csv       -     59 linhas
  ✓ C6\Fatura_2025-11-15.csv       -     65 linhas
  ✓ C6\Fatura_2025-12-15.csv       -     66 linhas
  ✓ C6\Fatura_2026-01-15.csv       -     43 linhas
  ✓ C6\Fatura_2026-03-15.csv       -     40 linhas
  ✓ C6\Fatura_2026-04-15.csv       -     62 linhas
  ✓ C6\Fatura_2026-05-15.csv       -     54 linhas
  ✓ C6\Fatura_2026-06-1

: 

In [10]:
import zipfile
import os

def descompactar_zips(pasta_origem, pasta_destino=None, senha=None):
    arquivos_zip = [f for f in os.listdir(pasta_origem) if f.endswith('.zip')]

    if not arquivos_zip:
        print("Nenhum arquivo ZIP encontrado.")
        return

    pwd = senha.encode() if isinstance(senha, str) else senha

    for arquivo in arquivos_zip:
        caminho_zip = os.path.join(pasta_origem, arquivo)
        destino = pasta_destino if pasta_destino else os.path.dirname(caminho_zip)

        os.makedirs(destino, exist_ok=True)

        try:
            with zipfile.ZipFile(caminho_zip, 'r') as zf:
                zf.extractall(destino, pwd=pwd)
                print(f"✔ {arquivo} → {destino}")
        except RuntimeError as e:
            print(f"✘ Erro em {arquivo}: {e}")

    print(f"\nConcluído!")

# Ou com destino customizado
descompactar_zips('../data/datasets/C6/', pasta_destino='../data/datasets/C6/', senha='405152')

✔ Fatura-CPF (1).zip → ../data/datasets/C6/
✔ Fatura-CPF (10).zip → ../data/datasets/C6/
✔ Fatura-CPF (11).zip → ../data/datasets/C6/
✔ Fatura-CPF (12).zip → ../data/datasets/C6/
✔ Fatura-CPF (13).zip → ../data/datasets/C6/
✔ Fatura-CPF (14).zip → ../data/datasets/C6/
✔ Fatura-CPF (15).zip → ../data/datasets/C6/
✔ Fatura-CPF (16).zip → ../data/datasets/C6/
✔ Fatura-CPF (17).zip → ../data/datasets/C6/
✔ Fatura-CPF (18).zip → ../data/datasets/C6/
✔ Fatura-CPF (19).zip → ../data/datasets/C6/
✔ Fatura-CPF (2).zip → ../data/datasets/C6/
✔ Fatura-CPF (20).zip → ../data/datasets/C6/
✔ Fatura-CPF (21).zip → ../data/datasets/C6/
✔ Fatura-CPF (22).zip → ../data/datasets/C6/
✔ Fatura-CPF (3).zip → ../data/datasets/C6/
✔ Fatura-CPF (4).zip → ../data/datasets/C6/
✔ Fatura-CPF (5).zip → ../data/datasets/C6/
✔ Fatura-CPF (6).zip → ../data/datasets/C6/
✔ Fatura-CPF (7).zip → ../data/datasets/C6/
✔ Fatura-CPF (8).zip → ../data/datasets/C6/
✔ Fatura-CPF (9).zip → ../data/datasets/C6/
✔ Fatura-CPF.zip → 

In [ ]:
combinar_csvs_simples('../data/datasets/nubank/', arquivo_saida='faturas_nubank.csv')

✅ Encontrados 26 arquivo(s) CSV

  ✓ nubank\Nubank_2024-01-20.csv   -      9 linhas
  ✓ nubank\Nubank_2024-02-20.csv   -     15 linhas
  ✓ nubank\Nubank_2024-03-20.csv   -     20 linhas
  ✓ nubank\Nubank_2024-04-20.csv   -     10 linhas
  ✓ nubank\Nubank_2024-05-20.csv   -     17 linhas
  ✓ nubank\Nubank_2024-06-20.csv   -     18 linhas
  ✓ nubank\Nubank_2024-07-20.csv   -     16 linhas
  ✓ nubank\Nubank_2024-08-20.csv   -     25 linhas
  ✓ nubank\Nubank_2024-09-20.csv   -     21 linhas
  ✓ nubank\Nubank_2024-10-20.csv   -     26 linhas
  ✓ nubank\Nubank_2024-11-20.csv   -     22 linhas
  ✓ nubank\Nubank_2024-12-20.csv   -     24 linhas
  ✓ nubank\Nubank_2025-02-20.csv   -     56 linhas
  ✓ nubank\Nubank_2025-03-20.csv   -     46 linhas
  ✓ nubank\Nubank_2025-05-20.csv   -     36 linhas
  ✓ nubank\Nubank_2025-06-20.csv   -     47 linhas
  ✓ nubank\Nubank_2025-08-20.csv   -     35 linhas
  ✓ nubank\Nubank_2025-09-08.csv   -     41 linhas
  ✓ nubank\Nubank_2025-10-08.csv   -     58 linha

,date,title,amount
0,2023-12-30,Tim*Tim,"40,00"
1,2023-12-29,Max,"30,00"
2,2023-12-20,Pag*Principia,"200,45"
3,2023-12-15,Pagamento recebido,"- 421,04"
4,2023-12-13,Ogura Pasteis,"20,00"
...,...,...,...
854,2026-06-01,Mp *Jhoonymorango,"20,00"
855,2026-06-01,Drogaria Sao Paulo - Parcela 2/2,"104,21"
856,2026-06-01,Yshpcienciada - Parcela 10/12,"43,57"
857,2026-06-01,Drogaria Sao Paulo - Parcela 2/2,"103,58"


: 

In [1]:
import pandas as pd

In [6]:
df = pd.read_csv('../data/datasets/gastosc6.csv', sep=';')

In [7]:
df.head()

,Data de Compra,Nome no Cartão,Final do Cartão,Categoria,Descrição,Parcela,Valor (em US$),Cotação (em R$),Valor (em R$)
0,08/12/2023,LUIZ HENRIQUE,5314.0,Empresa para empresa,RFM SUPER DE SORVETE,Única,0.0,0.0,73.49
1,09/12/2023,LUIZ HENRIQUE,5314.0,Transporte,UBER* TRIP,Única,0.0,0.0,19.99
2,09/12/2023,LUIZ HENRIQUE,5314.0,Restaurante / Lanchonete / Bar,IFOOD *IFD*MR DELIVE,Única,0.0,0.0,51.97
3,10/12/2023,LUIZ HENRIQUE,5314.0,Transporte,UBER* TRIP,Única,0.0,0.0,159.62
4,13/12/2023,LUIZ HENRIQUE,5314.0,Restaurante / Lanchonete / Bar,IFOOD *IFD*DIEGO ALV,Única,0.0,0.0,59.90


In [8]:
df['Categoria'].unique()

array(['Empresa para empresa', 'Transporte',
       'Restaurante / Lanchonete / Bar',
       'Supermercados / Mercearia / Padarias / Lojas de Conveniência',
       'Serviços pessoais', 'Assistência médica e odontológica',
       'Departamento / Desconto', 'Serviços Profissionais',
       'Entretenimento', nan, 'TV por assinatura / Serviços de rádio',
       '-', 'Empresa serviços', 'T&E',
       'Materiais de construção para casa', 'Recreativo', 'Associação',
       'Casa / Escritório Mobiliário', 'Serviços de telecomunicações',
       'Elétrico', 'Automotivo', 'Educacional', 'Especialidade varejo',
       'Marketing Direto', 'Vestuário / Roupas', 'Consertos em Geral',
       'Relacionados a Automotivo'], dtype=object)

In [29]:
from price_parser import Price
df = pd.read_csv('../data/datasets/Nubank_2026-08-08.csv')

df['amount'] = df['amount'].apply(lambda x: Price.fromstring(x).amount_float)
df = df[df['title'].str.lower().str.strip() != 'pagamento recebido']

In [30]:
df_limpo = df[df['amount'] > 0]
total = df_limpo['amount'].sum()
total

np.float64(1983.24)